# Module 2 — Dynamic Topology Extraction

Captures MLP activations, builds Pair Representations and Universal Modules, and exports the dynamic feature atlas.

In [ ]:
# Cell 1 – Dependency setup (circuit_sparsity injection + pip installs)
# version 1.11  (identical pattern to Module 1 Cell 1 v1.11)
import subprocess, sys, types, importlib

# ── pip installs ────────────────────────────────────────────────────────────
pkgs = ["h5py", "umap-learn", "seaborn", "transformers", "torch",
        "numpy", "pandas", "tqdm", "pyarrow"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── circuit_sparsity stub injection ─────────────────────────────────────────
# Must happen BEFORE any `from_pretrained` call for openai/circuit-sparsity
cs_pkg = types.ModuleType("circuit_sparsity")
cs_pkg.__path__ = []

# hook_utils stub
hu = types.ModuleType("circuit_sparsity.hook_utils")

def _hook_recorder(model, tokenizer, prompt, *, add_special_tokens=False, **kw):
    """Minimal fallback hook_recorder used when the real one is unavailable."""
    inputs = tokenizer(prompt, return_tensors="pt",
                       add_special_tokens=add_special_tokens)
    with __import__("torch").no_grad():
        out = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states  # tuple: (embedding, layer_0, ..., layer_N)
    acts = {}
    for i, h in enumerate(hidden[1:]):          # skip embedding layer
        acts[i] = h[0, -1, :].cpu()            # last-token, shape (d_model,)
    return acts

hu.hook_recorder = _hook_recorder
cs_pkg.hook_utils = hu

# gpt stub
gp = types.ModuleType("circuit_sparsity.gpt")
cs_pkg.gpt = gp

sys.modules["circuit_sparsity"] = cs_pkg
sys.modules["circuit_sparsity.hook_utils"] = hu
sys.modules["circuit_sparsity.gpt"] = gp

print("circuit_sparsity stubs injected")


In [ ]:
# Cell 2 – Configuration
# version 1.01

MODEL_ID       = "openai/circuit-sparsity"
PARQUET_PATH   = "/content/drive/MyDrive/CSP-Atlas/data/validated_prompts.parquet"
CHECKPOINT_DIR = "/content/drive/MyDrive/CSP-Atlas/checkpoints"
OUTPUT_HDF5    = "/content/drive/MyDrive/CSP-Atlas/dynamic_feature_atlas.h5"
STATS_JSON     = "/content/drive/MyDrive/CSP-Atlas/extraction_stats.json"

N_LAYERS            = 8        # circuitgpt has 8 transformer blocks
EPSILON             = 1e-3     # soft-to-hard binarization threshold
CONSISTENCY_THRESH  = 0.8      # fraction of variations neuron must fire in
CHECKPOINT_EVERY    = 200      # save checkpoint every N pairs
RESUME_CKPT         = None     # set to path string to resume

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Config OK")


In [ ]:
# Cell 3 – Imports
import sys, os, json, logging
import numpy as np
import pandas as pd
import torch
import h5py
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("module2")
print("Imports OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Cell 4 – Mount Drive & load data
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import pandas as pd
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df)} rows | "
      f"{df.groupby(['ast_node','builtin_obj']).ngroups} unique pairs")
df.head(3)


In [ ]:
# Cell 5 – Load model & tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float32,
).to(DEVICE).eval()

print(f"Model loaded on {DEVICE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

# Import hook_recorder from the (possibly stubbed) module
from circuit_sparsity.hook_utils import hook_recorder
print("hook_recorder available:", callable(hook_recorder))


In [ ]:
# Cell 6 – ActivationExtractor sanity check
import sys
sys.path.insert(0, "/Users/piotrwilam/Code/CSP-Atlas/src")  # local dev path
# On Colab: clone repo and adjust path, or copy src/ to /content/

from module2.extraction import ActivationExtractor

extractor = ActivationExtractor(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    n_layers=N_LAYERS,
    use_hook_recorder=True,
    hook_recorder_fn=hook_recorder,
)

# Quick diagnostic: extract one prompt
sample_prompt = df["prompt_text"].iloc[0]
acts = extractor.extract(sample_prompt)
print(f"Extracted {len(acts)} layers")
for lid, vec in sorted(acts.items()):
    print(f"  Layer {lid}: shape={vec.shape}, "
          f"active={(vec.abs() > EPSILON).sum().item()} / {vec.numel()}")


In [ ]:
# Cell 7 – PairRepresentationBuilder sanity check
from module2.binarization import PairRepresentationBuilder

builder = PairRepresentationBuilder(
    epsilon=EPSILON,
    consistency_threshold=CONSISTENCY_THRESH,
    n_layers=N_LAYERS,
)

# Test on first pair
first_pair = df.groupby(["ast_node", "builtin_obj"]).groups
first_key  = list(first_pair.keys())[0]
sample_prompts = df.loc[first_pair[first_key], "prompt_text"].tolist()

pair_masks_sample = builder.build(extractor, sample_prompts[:3])
print(f"Pair {first_key}: {len(pair_masks_sample)} layers")
for lid, m in sorted(pair_masks_sample.items()):
    print(f"  Layer {lid}: circuit_size={m.sum()} / {m.size}")


In [ ]:
# Cell 8 – Full pipeline run (with checkpointing)
from module2.pipeline import Module2Pipeline

pipeline = Module2Pipeline(
    extractor=extractor,
    builder=builder,
    parquet_path=PARQUET_PATH,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=CHECKPOINT_EVERY,
)

pair_masks, universal_masks, metrics, stats_df = pipeline.run(
    resume_from_checkpoint=RESUME_CKPT
)

print(f"Pairs extracted : {len(pair_masks)}")
print(f"Universal AST   : {len(universal_masks['ast'])}")
print(f"Universal Builtin: {len(universal_masks['builtin'])}")
stats_df.head()


In [ ]:
# Cell 9 – Inspect circuit sizes
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of circuit sizes across all pairs and layers
sizes = stats_df["circuit_size"].values
plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=50, edgecolor="black")
plt.xlabel("Circuit size (# active neurons)")
plt.ylabel("Count")
plt.title("Distribution of Pair Representation circuit sizes")
plt.tight_layout()
plt.show()

print(f"Mean circuit size: {sizes.mean():.1f} ± {sizes.std():.1f}")
print(f"Min: {sizes.min()} | Max: {sizes.max()}")


In [ ]:
# Cell 10 – Universal module stats
ast_sizes    = {}
builtin_sizes = {}

rep_layer = 4  # representative layer for inspection

for name, layers in universal_masks["ast"].items():
    if rep_layer in layers:
        ast_sizes[name] = int(layers[rep_layer].sum())

for name, layers in universal_masks["builtin"].items():
    if rep_layer in layers:
        builtin_sizes[name] = int(layers[rep_layer].sum())

print(f"Universal AST modules with data at layer {rep_layer}: {len(ast_sizes)}")
print(f"Universal Builtin modules with data at layer {rep_layer}: {len(builtin_sizes)}")

# Top-10 largest
top_ast = sorted(ast_sizes.items(), key=lambda x: -x[1])[:10]
print("\nTop 10 Universal AST circuits:")
for n, s in top_ast:
    print(f"  {n}: {s} neurons")


In [ ]:
# Cell 11 – Jaccard matrix diagnostics
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if "jaccard_ast_matrix" in metrics:
    mat   = metrics["jaccard_ast_matrix"]
    names = metrics.get("ast_names", [str(i) for i in range(mat.shape[0])])

    fig, ax = plt.subplots(figsize=(min(20, len(names)*0.4+2),
                                    min(20, len(names)*0.4+2)))
    sns.heatmap(mat, ax=ax, vmin=0, vmax=1,
                xticklabels=names if len(names) <= 30 else False,
                yticklabels=names if len(names) <= 30 else False,
                cmap="viridis")
    ax.set_title("Jaccard Similarity — Universal AST Modules (layer 4)")
    plt.tight_layout()
    plt.show()

    off_diag = mat[np.triu_indices_from(mat, k=1)]
    print(f"Mean off-diag Jaccard: {off_diag.mean():.4f}")
    print(f"Max  off-diag Jaccard: {off_diag.max():.4f}")


In [ ]:
# Cell 12 – Entanglement Index spot check
from module2.metrics import entanglement_index

ei_results = []
rep_layer  = 4

for (ast_n, blt_o), layers in pair_masks.items():
    if rep_layer not in layers:
        continue
    pm = layers[rep_layer]
    am = universal_masks["ast"].get(ast_n, {}).get(rep_layer)
    bm = universal_masks["builtin"].get(blt_o, {}).get(rep_layer)
    if am is None or bm is None:
        continue
    ei = entanglement_index(pm, am, bm)
    ei_results.append({"ast_node": ast_n, "builtin_obj": blt_o, "E_I": ei})

ei_df = pd.DataFrame(ei_results)
print(f"Entanglement Index — {len(ei_df)} pairs at layer {rep_layer}")
print(ei_df["E_I"].describe())
ei_df.sort_values("E_I", ascending=False).head(10)


In [ ]:
# Cell 13 – Layer-wise circuit evolution
layer_ids = sorted({lid for layers in pair_masks.values() for lid in layers})

mean_per_layer = []
for lid in layer_ids:
    sizes = [layers[lid].sum()
             for layers in pair_masks.values() if lid in layers]
    mean_per_layer.append(np.mean(sizes) if sizes else 0)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(layer_ids, mean_per_layer, marker="o")
plt.xlabel("Layer")
plt.ylabel("Mean circuit size (neurons)")
plt.title("Mean Pair Representation circuit size vs. layer")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 14 – Save HDF5 atlas
import sys, os
sys.path.insert(0, "/Users/piotrwilam/Code/CSP-Atlas/src")

from module2.io_utils import save_atlas_hdf5

metadata = {
    "model_id"           : MODEL_ID,
    "n_layers"           : N_LAYERS,
    "epsilon"            : EPSILON,
    "consistency_thresh" : CONSISTENCY_THRESH,
    "n_pairs"            : len(pair_masks),
    "ast_nodes"          : sorted(set(a for a, _ in pair_masks)),
    "builtin_objs"       : sorted(set(b for _, b in pair_masks)),
}

save_atlas_hdf5(OUTPUT_HDF5, pair_masks, universal_masks, metrics, metadata)
print("Atlas saved:", OUTPUT_HDF5)


In [ ]:
# Cell 15 – Save extraction stats JSON
import json

stats_out = {
    "n_pairs"        : len(pair_masks),
    "n_universal_ast": len(universal_masks["ast"]),
    "n_universal_blt": len(universal_masks["builtin"]),
    "circuit_size_mean": float(stats_df["circuit_size"].mean()),
    "circuit_size_std" : float(stats_df["circuit_size"].std()),
}

with open(STATS_JSON, "w") as fh:
    json.dump(stats_out, fh, indent=2)
print("Stats saved:", STATS_JSON)
print(json.dumps(stats_out, indent=2))


In [ ]:
# Cell 16 – Verify HDF5 round-trip
import h5py

with h5py.File(OUTPUT_HDF5, "r") as f:
    def _walk(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: {obj.shape} {obj.dtype}")
    f.visititems(_walk)
    print("\nMetadata attrs:", dict(f["metadata"].attrs))


In [ ]:
# Cell 17 – Done
print("Module 2 extraction complete.")
print(f"  Atlas  : {OUTPUT_HDF5}")
print(f"  Stats  : {STATS_JSON}")
print(f"  Pairs  : {len(pair_masks)}")
print(f"  Layers : {N_LAYERS}")
